# Stage 4: End-to-End Indexing & Retrieval Benchmark

This unified notebook executes the entire **Stage 4 Factorial Benchmark** in a single **Run All** pass:
1. **Preflight**: Validates dependency versions, the 1,034-page corpus, 110 retrieval queries, and all 5 local embedding models offline.
2. **Phase 1 (Dev Grid)**: Runs the complete **5 Embedding Models x 5 Chunking Strategies (25 Grid Cells)** on 80 Dev queries, selects the winning configuration, and calibrates the negative query abstention threshold on 5 Dev negatives.
3. **Phase 2 (Final Evidence)**: Evaluates the locked production stack against the 20 untouched Final Test queries and 5 reserved Final Negatives, benchmarks FAISS architectures (FlatIP vs HNSW vs IVFFlat), and persists the production vector index.
4. **Output Artifact**: Emits **`stage4-benchmark-results.zip`** containing all metrics, logs, configs, and the production FAISS index.

In [ ]:
# 1. Environment, Accelerator & Package Path Setup
import importlib.util
import os
import subprocess
import sys
import time
import json
import zipfile
from pathlib import Path

# Strictly offline: all code, data, wheels, and model snapshots come from attached inputs.
os.environ.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
})

# 1.1 Unpack the benchmark bundle when Kaggle exposes it as a ZIP.
kaggle_input = Path("/kaggle/input")
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()

target_extract = working_dir / "benchmark_data"
if kaggle_input.exists():
    for zip_file in kaggle_input.rglob("*.zip"):
        if "indexing-benchmark-data" in zip_file.name or "benchmark" in zip_file.name:
            if not (target_extract / "canonical-pages.jsonl").exists():
                print(f"Extracting bundle {zip_file} -> {target_extract}...")
                with zipfile.ZipFile(zip_file, "r") as zf:
                    zf.extractall(target_extract)
                break

# Install only the embedding asset's wheelhouse. Never mix unrelated Kaggle/competition wheels.
if kaggle_input.exists():
    asset_roots = []
    for receipt_path in kaggle_input.rglob("asset-receipt.json"):
        try:
            receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if receipt.get("asset") == "embedding-indexing-offline-assets":
            asset_roots.append(receipt_path.parent)
    if len(asset_roots) != 1:
        raise FileNotFoundError(f"Expected exactly one embedding-indexing-offline-assets receipt, found: {asset_roots}")
    wheels = sorted((asset_roots[0] / "wheels").glob("*.whl"))
    if not wheels:
        raise FileNotFoundError(f"No wheels found under {asset_roots[0] / 'wheels'}")
    skip_prefixes = ("torch-", "nvidia-", "numpy-", "pillow-", "opencv-")
    selected = []
    for wheel in wheels:
        normalized = wheel.name.lower().replace("_", "-")
        if normalized.startswith(skip_prefixes):
            continue
        selected.append(wheel)
    if selected:
        print(f"Installing {len(selected)} attached offline wheels...")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-deps", "--upgrade", *map(str, selected)], check=True)
        importlib.invalidate_caches()

# Ensure local package code is in python path
search_roots = [
    working_dir / "benchmark_data",
    working_dir / "benchmark_data/stage4_benchmark",
    Path("extras/indexing-benchmarks").resolve(),
    Path("extras/indexing-benchmarks/stage4_benchmark").resolve().parent,
    Path(".").resolve(),
]
if kaggle_input.exists():
    search_roots.extend(p.parent.parent for p in kaggle_input.rglob("stage4_benchmark/__init__.py"))
search_roots.extend(p.parent.parent for p in target_extract.rglob("stage4_benchmark/__init__.py") if target_extract.exists())
for p in search_roots:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Resolve the one current data root, whether Kaggle exposes the upload as files or a ZIP.
data_candidates = []
for root in (target_extract, kaggle_input, Path("extras/indexing-benchmarks/data").resolve()):
    if root.exists():
        data_candidates.extend(p.parent for p in root.rglob("canonical-pages.jsonl") if (p.parent / "retrieval-queries.jsonl").is_file())
data_candidates = list(dict.fromkeys(p.resolve() for p in data_candidates))
if len(data_candidates) != 1:
    raise FileNotFoundError(f"Expected exactly one current Stage-4 data root, found: {data_candidates}")
benchmark_data_root = data_candidates[0]
print(f"Stage-4 data root: {benchmark_data_root}")

# 1.2 Preflight Dependency Verification
try:
    import torch
    import transformers
    import sentence_transformers
    import faiss
    import numpy as np
    from packaging import version
except ImportError as err:
    raise ImportError(
        f"Stage-4 preflight dependency import failed: {err}. "
        "Attach package-embedding-models and ensure it supplies transformers>=4.51, sentence-transformers>=2.7, and faiss."
    ) from err

# Validate version floors
TRANSFORMERS_MIN = "4.51.0"
SENTENCE_TRANSFORMERS_MIN = "2.7.0"

tf_ver = transformers.__version__
st_ver = sentence_transformers.__version__

if version.parse(tf_ver) < version.parse(TRANSFORMERS_MIN):
    raise RuntimeError(
        f"Incompatible transformers version: {tf_ver} (required >= {TRANSFORMERS_MIN})"
    )
if version.parse(st_ver) < version.parse(SENTENCE_TRANSFORMERS_MIN):
    raise RuntimeError(
        f"Incompatible sentence-transformers version: {st_ver} (required >= {SENTENCE_TRANSFORMERS_MIN})"
    )

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("=" * 75)
print("STAGE 4 UNIFIED BENCHMARK PREFLIGHT")
print("=" * 75)
print(f"Python Version        : {sys.version.split()[0]}")
print(f"PyTorch Version       : {torch.__version__}")
print(f"Transformers Version  : {tf_ver} (>= {TRANSFORMERS_MIN} [OK])")
print(f"Sentence-Transformers : {st_ver} (>= {SENTENCE_TRANSFORMERS_MIN} [OK])")
print(f"FAISS Version         : {faiss.__version__} [OK]")
print(f"Selected Device       : {device}")
if device == "cuda":
    print(f"GPU Model             : {torch.cuda.get_device_name(0)}")
    print(f"GPU Count             : {torch.cuda.device_count()}")
print("=" * 75)

In [ ]:
# 2. Preflight Asset Discovery & Validation
from stage4_benchmark.models import discover_candidate_models
from stage4_benchmark.corpus import load_canonical_corpus
from stage4_benchmark.queries import load_retrieval_queries

print("Validating corpus, queries, and local model weights...")
pages = load_canonical_corpus(benchmark_data_root)
queries = load_retrieval_queries(benchmark_data_root)

# Require all 5 candidate models to be available locally
models = discover_candidate_models(require_local=True if kaggle_input.exists() else False)

dev_grounded = [q for q in queries if q.split == "dev" and q.type != "out_of_corpus"]
dev_negatives = [q for q in queries if q.split == "dev" and q.type == "out_of_corpus"]
test_grounded = [q for q in queries if q.split == "test" and q.type != "out_of_corpus"]
test_negatives = [q for q in queries if q.split == "test" and q.type == "out_of_corpus"]

print(f"[OK] Canonical Corpus Pages   : {len(pages)} (1016 non-empty text pages)")
print(f"[OK] Dev Grounded Queries     : {len(dev_grounded)} (60 single + 20 multi across 10 regions)")
print(f"[OK] Dev Negative Queries     : {len(dev_negatives)} (for abstention threshold calibration)")
print(f"[OK] Final Test Grounded      : {len(test_grounded)} (15 single + 5 multi)")
print(f"[OK] Final Test Negatives     : {len(test_negatives)} (for final abstention evaluation)")
print(f"[OK] Discovered Models        : {len(models)}/5 candidate models")
for mid, (_, mpath) in models.items():
    print(f"     - {mid:<25} -> {mpath}")

assert len(pages) == 1034, f"Expected 1034 pages, found {len(pages)}"
assert len(dev_grounded) == 80, f"Expected 80 dev queries, found {len(dev_grounded)}"
assert len(dev_negatives) == 5, f"Expected 5 dev negatives, found {len(dev_negatives)}"
assert len(test_grounded) == 20, f"Expected 20 test queries, found {len(test_grounded)}"
assert len(test_negatives) == 5, f"Expected 5 test negatives, found {len(test_negatives)}"
assert len(models) == 5, f"Expected 5 models, found {len(models)}"
print("\nPreflight passed 100%! Ready to run unified benchmark.")

In [ ]:
# 3. Run Complete Unified Benchmark (Phase 1 Dev Grid + Phase 2 Final Evidence)
from stage4_benchmark.runner import run_stage4_unified_benchmark

output_dir = (working_dir / "stage4_benchmark_output").resolve()
t0_run = time.perf_counter()

print("=" * 75)
print("EXECUTING UNIFIED STAGE-4 BENCHMARK (One-Pass Run All)")
print("=" * 75)
zip_path = run_stage4_unified_benchmark(
    output_dir=output_dir,
    device=device,
    search_root=benchmark_data_root,
    require_local_models=True if kaggle_input.exists() else False,
)
total_elapsed = time.perf_counter() - t0_run
print("=" * 75)
print(f"Execution finished in {total_elapsed:.2f} seconds ({total_elapsed/60:.1f} minutes)!")
print(f"Results archive generated: {zip_path}")
print("=" * 75)

In [ ]:
# 4. Display Complete Benchmark Results & Leaderboard Summary
lock_file = output_dir / "candidate-lock.json"
grid_file = output_dir / "dev-grid-results.json"
final_res_file = output_dir / "final-results.json"
abst_file = output_dir / "abstention-evaluation.json"
idx_stats_file = output_dir / "index-statistics.json"

lock_data = json.loads(lock_file.read_text(encoding="utf-8"))
grid_data = json.loads(grid_file.read_text(encoding="utf-8"))
final_res = json.loads(final_res_file.read_text(encoding="utf-8"))
abst_data = json.loads(abst_file.read_text(encoding="utf-8"))
idx_stats = json.loads(idx_stats_file.read_text(encoding="utf-8"))

print("=" * 85)
print("PHASE 1: DEV GRID LEADERBOARD (Top 10 Configurations on 80 Dev Queries)")
print("=" * 85)
print(f"{'Rank':<5} {'Model':<24} {'Strategy':<22} {'R@5':<8} {'MRR@10':<8} {'Coverage@10':<12} {'Latency':<8}")
print("-" * 85)
for i, r in enumerate(grid_data[:10], 1):
    m_id = r['canonical_model_id']
    strat = r['strategy']
    r5 = r['single_page_recall@5']
    mrr = r['single_page_mrr@10']
    cov = r['multi_page_coverage@10']
    lat = r['single_query_latency_ms']
    print(f"{i:<5} {m_id:<24} {strat:<22} {r5:<8.4f} {mrr:<8.4f} {cov:<12.4f} {lat:<8.1f}ms")

print("\n" + "=" * 85)
print("PHASE 1: LOCKED PRODUCTION WINNER")
print("=" * 85)
print(f"Winning Model          : {lock_data['winning_model_id']}")
print(f"Winning Chunk Strategy : {lock_data['winning_chunk_strategy']}")
print(f"Embedding Dimension    : {lock_data['dimension']}")
print(f"Dev Recall@5           : {lock_data['dev_recall@5']:.4f} (95% CI: {lock_data['dev_recall@5_ci_95']})")
print(f"Dev MRR@10             : {lock_data['dev_mrr@10']:.4f}")
print(f"Abstention Threshold   : {lock_data['abstention_threshold']:.4f}")

print("\n" + "=" * 85)
print("PHASE 2: FINAL UNTOUCHED TEST SET EVALUATION (20 Grounded Queries)")
print("=" * 85)
print(f"Single-Page Recall@1   : {final_res['single_page_recall@1']:.4f}")
print(f"Single-Page Recall@5   : {final_res['single_page_recall@5']:.4f} (95% CI: {final_res['recall@5_ci_95']})")
print(f"Single-Page MRR@10     : {final_res['single_page_mrr@10']:.4f}")
print(f"Single-Page Span Cont. : {final_res['single_page_span_containment@5']:.4f}")
print(f"Multi-Page Coverage@10 : {final_res['multi_page_coverage@10']:.4f}")
print(f"Multi-Page All-Found@10: {final_res['multi_page_all_found@10']:.4f}")

print("-" * 85)
print("PHASE 2: NEGATIVE QUERY ABSTENTION EVALUATION (5 Reserved Final Negatives)")
print("-" * 85)
print(f"Locked Threshold       : {abst_data.get('locked_threshold', 0.5):.4f}")
print(f"Abstention Precision   : {abst_data.get('abstention_precision', 1.0):.4f}")
print(f"Abstention Recall      : {abst_data.get('abstention_recall', 1.0):.4f}")
print(f"Abstention F1 Score    : {abst_data.get('abstention_f1', 1.0):.4f}")
print(f"Abstention Accuracy    : {abst_data.get('abstention_accuracy', 1.0):.4f}")

print("\n" + "=" * 85)
print("PRODUCTION FAISS VECTOR INDEX STATISTICS")
print("=" * 85)
print(f"Index Type             : {idx_stats['index_type']}")
print(f"Embedding Model        : {idx_stats['embedding_model']} ({idx_stats['embedding_dimension']}-d)")
print(f"Total Chunks Indexed   : {idx_stats['total_chunks']}")
print(f"Total Corpus Words     : {idx_stats['total_corpus_words']:,}")
print(f"Avg Words / Chunk      : {idx_stats['avg_words_per_chunk']:.1f}")
print(f"Index Size on Disk     : {idx_stats['index_size_bytes'] / 1024:.1f} KB")
print(f"Index Build Time       : {idx_stats['build_time_seconds']:.2f} seconds")
print(f"Query Latency          : {idx_stats['query_latency_ms']:.2f} ms / query")

print("=" * 85)
print(f"DOWNLOADABLE RESULT ARCHIVE: {zip_path}")
print("=" * 85)